# 02 Data Cleaning and EDA

KNHANES 2017-2023 ???? ??? ???, 19-39? ???? ??? ? ??? target? ???.

## 1. Library import

In [ ]:
# If this cell gives an import error, run the install cell below first.
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Run only if needed
# !pip install pyreadstat seaborn openpyxl

In [ ]:
import seaborn as sns

pd.set_option('display.max_columns', 50)
plt.style.use('default')

## 2. Load SAV files

?? `.sav` ??? GitHub? ??? ??, local folder `???????` ?? ??.

In [ ]:
data_dir = Path('???????')
sav_files = sorted(data_dir.glob('HN*_all.sav'))

print('Number of SAV files:', len(sav_files))
for file in sav_files:
    print(file.name)

## 3. Select important columns

?? ?????? ??? ?? ??? ?? ????.

In [ ]:
important_cols = [
    'age', 'sex',
    'HE_BMI', 'HE_glu', 'HE_TG',
    'sm_presnt', 'dr_month', 'pa_aerobic',
    'incm', 'ho_incm', 'educ',
    'HE_sbp', 'HE_dbp'
]

In [ ]:
all_data = []

for file in sav_files:
    year = 2000 + int(file.name[2:4])
    print('Loading:', file.name, 'year =', year)
    
    df = pd.read_spss(file)
    available_cols = [col for col in important_cols if col in df.columns]
    temp = df[available_cols].copy()
    temp['year'] = year
    
    all_data.append(temp)
    print('  shape:', temp.shape)

knhanes = pd.concat(all_data, ignore_index=True)
print('Combined shape:', knhanes.shape)
knhanes.head()

## 4. COVID period ???

In [ ]:
def make_covid_period(year):
    if year <= 2019:
        return 'before'
    elif year <= 2021:
        return 'during'
    else:
        return 'after'

knhanes['covid_period'] = knhanes['year'].apply(make_covid_period)
knhanes[['year', 'covid_period']].head()

## 5. Filter young adults: age 19-39

In [ ]:
young = knhanes[(knhanes['age'] >= 19) & (knhanes['age'] <= 39)].copy()

print('Before filtering:', knhanes.shape)
print('After filtering age 19-39:', young.shape)
young['age'].describe()

## 6. Hypertension target ???

??? target? SBP ?? DBP ???? ???.

- hypertension = 1: HE_sbp >= 140 ?? HE_dbp >= 90
- hypertension = 0: ? ?

??: HE_sbp, HE_dbp? target? ?? ?? ???? ?? feature??? ????.

In [ ]:
young['hypertension'] = np.where(
    (young['HE_sbp'] >= 140) | (young['HE_dbp'] >= 90),
    1,
    0
)

young['hypertension'].value_counts(dropna=False)

## 7. Missing value check

In [ ]:
missing_table = young.isna().sum().sort_values(ascending=False)
missing_table = missing_table[missing_table > 0]
missing_table

In [ ]:
# percent missing
missing_percent = (young.isna().mean() * 100).sort_values(ascending=False)
missing_percent = missing_percent[missing_percent > 0]
missing_percent

## 8. Basic EDA

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=young, x='year')
plt.title('Number of participants by year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=young, x='covid_period', order=['before', 'during', 'after'])
plt.title('Number of participants by COVID period')
plt.xlabel('COVID period')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=young, x='hypertension')
plt.title('Hypertension target distribution')
plt.xlabel('Hypertension')
plt.ylabel('Count')
plt.show()

print(young['hypertension'].value_counts(normalize=True))

In [ ]:
period_htn = pd.crosstab(young['covid_period'], young['hypertension'], normalize='index')
period_htn = period_htn.loc[['before', 'during', 'after']]
period_htn

In [ ]:
period_htn.plot(kind='bar', figsize=(7, 4))
plt.title('Hypertension ratio by COVID period')
plt.xlabel('COVID period')
plt.ylabel('Ratio')
plt.xticks(rotation=0)
plt.legend(title='Hypertension')
plt.show()

## 9. Numeric variable distributions

In [ ]:
numeric_cols = ['age', 'HE_BMI', 'HE_glu', 'HE_TG']

young[numeric_cols].describe()

In [ ]:
for col in ['HE_BMI', 'HE_glu', 'HE_TG']:
    plt.figure(figsize=(6, 4))
    sns.histplot(data=young, x=col, bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

In [ ]:
for col in ['HE_BMI', 'HE_glu', 'HE_TG']:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=young, x='hypertension', y=col)
    plt.title(f'{col} by hypertension')
    plt.xlabel('Hypertension')
    plt.ylabel(col)
    plt.show()

## 10. Categorical variables

In [ ]:
categorical_cols = ['sex', 'sm_presnt', 'dr_month', 'pa_aerobic', 'incm', 'ho_incm', 'educ', 'covid_period']

for col in categorical_cols:
    if col in young.columns:
        print('
', col)
        print(young[col].value_counts(dropna=False).head(10))

## 11. Save cleaned dataset

??? notebook?? ?? ??? ? ??? cleaned CSV ??? ????. ?? `.sav` ??? ???? ???.

In [ ]:
output_dir = Path('processed_data')
output_dir.mkdir(exist_ok=True)

cleaned_path = output_dir / 'knhanes_young_hypertension_2017_2023.csv'
young.to_csv(cleaned_path, index=False, encoding='utf-8-sig')

print('Saved:', cleaned_path)
print('Shape:', young.shape)